# 📄 PDF Compressor

Compress multiple PDF files using Ghostscript in Google Colab.

### Features
- Upload multiple PDF files
- Choose Low, Medium, or High compression
- Show original and compressed sizes
- Calculate compression percentage
- Create a ZIP file containing all compressed PDFs
- Automatically download the ZIP file

## 1. Install Ghostscript

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ghostscript

## 2. Upload Multiple PDFs

In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith('.pdf')
]

print(f'Uploaded {len(pdf_files)} PDF file(s):')

for pdf_file in pdf_files:
    print(f'  - {pdf_file}')

## 3. Select Compression Level

Choose one of:

- `low` - High quality, larger file
- `medium` - Balanced quality and size
- `high` - Smaller file, lower image quality

In [ ]:
COMPRESSION_LEVEL = 'medium'

compression_settings = {
    'low': '/printer',
    'medium': '/ebook',
    'high': '/screen'
}

if COMPRESSION_LEVEL not in compression_settings:
    raise ValueError("Choose: low, medium, or high")

ghostscript_setting = compression_settings[COMPRESSION_LEVEL]

print(f'Compression level: {COMPRESSION_LEVEL.upper()}')
print(f'Ghostscript preset: {ghostscript_setting}')

## 4. Compress PDFs

In [ ]:
import os
import subprocess

output_folder = 'compressed_pdfs'
os.makedirs(output_folder, exist_ok=True)

compressed_files = []

for pdf_file in pdf_files:
    base_name = os.path.splitext(os.path.basename(pdf_file))[0]
    output_file = os.path.join(
        output_folder,
        f'{base_name}_compressed.pdf'
    )

    print(f'Compressing: {pdf_file}')

    command = [
        'gs',
        '-sDEVICE=pdfwrite',
        '-dCompatibilityLevel=1.4',
        f'-dPDFSETTINGS={ghostscript_setting}',
        '-dNOPAUSE',
        '-dQUIET',
        '-dBATCH',
        f'-sOutputFile={output_file}',
        pdf_file
    ]

    try:
        subprocess.run(
            command,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )

        if os.path.exists(output_file):
            original_size = os.path.getsize(pdf_file)
            compressed_size = os.path.getsize(output_file)

            if original_size > 0:
                reduction = (1 - compressed_size / original_size) * 100
            else:
                reduction = 0

            compressed_files.append(output_file)

            print(f'  ✓ Saved: {output_file}')
            print(f'  Original:   {original_size / 1024 / 1024:.2f} MB')
            print(f'  Compressed: {compressed_size / 1024 / 1024:.2f} MB')
            print(f'  Reduction:  {reduction:.1f}%')
            print()

    except subprocess.CalledProcessError as error:
        print(f'  ✗ Failed: {pdf_file}')
        print(error.stderr.decode('utf-8', errors='ignore'))

print(f'Finished. Compressed {len(compressed_files)} PDF file(s).')

## 5. Create ZIP File

In [ ]:
import zipfile

zip_name = f'compressed_pdfs_{COMPRESSION_LEVEL}.zip'

with zipfile.ZipFile(
    zip_name,
    'w',
    zipfile.ZIP_DEFLATED
) as zipf:
    for compressed_file in compressed_files:
        zipf.write(
            compressed_file,
            arcname=os.path.basename(compressed_file)
        )

print(f'✓ Created: {zip_name}')
print(f'✓ Contains {len(compressed_files)} PDF file(s)')

## 6. Automatically Download ZIP

In [ ]:
from google.colab import files

if compressed_files:
    files.download(zip_name)
else:
    print('No compressed files available for download.')

## Compression Levels

| Level | Quality | File Size | Recommended For |
|---|---|---|---|
| 🟢 Low | High | Larger | Printing and archiving |
| 🟡 Medium | Good | Medium | General use |
| 🔴 High | Lower | Smaller | Email and web |

### Change the compression level

Edit this line in Cell 3:

```python
COMPRESSION_LEVEL = 'medium'
```

Available options:

```text
low
medium
high
```